# 文件二：高级调优实践

一. **初始化**  
  - 1.1 设置 Catalog 和 Schema  
  - 1.2 验证表是否存在  
  - 1.3 创建辅助表 – 按天/年分区（用于对比）

二. **分区策略对比测试**  
  - 2.1 对比不同分区粒度的文件数量  
  - 2.2 对比不同分区粒度的查询性能（相同日期范围查询）  
  - 注：分区裁剪只对分区键生效，不能依赖派生字段

三. **存储格式转换对比**  
  - 3.1 创建和写入 ORC 文件  
  - 3.2 对比文件大小（Delta/Parquet vs ORC）  
  - 3.3 不同文件格式查询效率对比（ORC vs Delta）

四. **数据倾斜与处理**  
  - 4.1 检查数据倾斜程度（百分位数、Top 客户）  
  - 4.2 测试倾斜情况下的查询效率（普通 GROUP BY）  
  - 4.3 两阶段聚合优化（加随机前缀打散 key）  
  - 4.4 对比优化前后执行计划（分析无效/负优化）  
  - 4.5 MapJoin 优化示例（订单表 join 产品表）

In [0]:
%python
#一.初始化
# 1.1 设置 Catalog 和 Schema
spark.sql("USE CATALOG data_catalog")
spark.sql("USE SCHEMA retail_data")

# 1.2 验证表是否存在（如果不存在，请运行前一个脚本以生成所需的表）
tables = spark.sql("SHOW TABLES").collect()
for t in tables:
    print(f"{t.tableName}")

ads_customer_activity
demo_optimize
dim_customer
dim_product
dws_customer_month_summary
fact_order_day_partitioned
fact_order_month_partitioned
fact_order_partitioned
fact_order_raw
fact_order_year_partitioned


In [0]:
%python
# 1.3 创建辅助表 – 按天/年分区
# 注意：按天分区会产生大量小文件，此处仅用于演示
# 按天分区表
spark.sql("""
    CREATE OR REPLACE TABLE fact_order_day_partitioned
    USING DELTA
    PARTITIONED BY (order_date)
    AS SELECT * FROM fact_order_raw
""")

# 按年分区表
spark.sql("""
    CREATE OR REPLACE TABLE fact_order_year_partitioned
    USING DELTA
    PARTITIONED BY (year_key)
    AS 
    SELECT *, YEAR(order_date) AS year_key
    FROM fact_order_raw
""")

print("已创建按天分区表和按年分区表，用于对比")

已创建按天分区表和按年分区表，用于对比


In [0]:
%python
# 二.分区策略对比测试
# 2.1 对比不同分区粒度的文件数量
# 按天应该最多，月份次之，年份最少
from pprint import pprint

def get_file_count(table_name):
    detail = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]
    return detail.numFiles

tables = ["fact_order_day_partitioned", "fact_order_month_partitioned", "fact_order_year_partitioned"]
for t in tables:
    cnt = get_file_count(t)
    print(f"{t}: {cnt} 个文件")

fact_order_day_partitioned: 4018 个文件
fact_order_month_partitioned: 132 个文件
fact_order_year_partitioned: 11 个文件


In [0]:
%python
# 2.2 对比不同分区粒度的查询性能（相同日期范围查询）
#注意，分区裁剪只对分区键本身的过滤条件生效，不能依赖派生字段。
#在设计分区策略时，必须确保常用查询条件直接使用分区键，否则分区无效。
#预期结果：按天最慢，按月次之，按年最快
import time
import statistics

# 按天分区（使用 order_date 过滤，自然裁剪）
start = time.time()
query_day = """
    SELECT customer_id, SUM(amount) AS total
    FROM fact_order_day_partitioned
    WHERE order_date >= '2020-01-01' AND order_date <= '2020-12-31'
    GROUP BY customer_id
    ORDER BY total DESC
    LIMIT 10
"""
spark.sql(query_day).show()
end = time.time()
partition_date_time = end - start

# 按月分区（使用 month_key 过滤，裁剪到12个分区）
start = time.time()
query_month = """
    SELECT customer_id, SUM(amount) AS total
    FROM fact_order_month_partitioned
    WHERE month_key BETWEEN '2020-01' AND '2020-12'
    GROUP BY customer_id
    ORDER BY total DESC
    LIMIT 10
"""
spark.sql(query_month).show()
end = time.time()
partition_month_time = end - start

# 按年分区（使用 year_key 过滤，裁剪到1个分区）
start = time.time()
query_year = """
    SELECT customer_id, SUM(amount) AS total
    FROM fact_order_year_partitioned
    WHERE year_key = 2020
    GROUP BY customer_id
    ORDER BY total DESC
    LIMIT 10
"""
spark.sql(query_year).show()
end = time.time()
partition_year_time = end - start

#输出：
print(f"按天分区耗时: {partition_date_time:.2f} 秒")
print(f"按月分区耗时: {partition_month_time:.2f} 秒")
print(f"按年分区耗时: {partition_year_time:.2f} 秒")


+-----------+--------+
|customer_id|   total|
+-----------+--------+
|    13025BA|21838966|
|    13026BA|21827791|
|    13028BA|21702702|
|    13023BA|21604209|
|    13029BA|21596067|
|    13021BA|21569177|
|    13030BA|21542906|
|    13024BA|21483391|
|    13022BA|21398939|
|    13027BA|21375556|
+-----------+--------+

+-----------+--------+
|customer_id|   total|
+-----------+--------+
|    13025BA|21838966|
|    13026BA|21827791|
|    13028BA|21702702|
|    13023BA|21604209|
|    13029BA|21596067|
|    13021BA|21569177|
|    13030BA|21542906|
|    13024BA|21483391|
|    13022BA|21398939|
|    13027BA|21375556|
+-----------+--------+

+-----------+--------+
|customer_id|   total|
+-----------+--------+
|    13025BA|21838966|
|    13026BA|21827791|
|    13028BA|21702702|
|    13023BA|21604209|
|    13029BA|21596067|
|    13021BA|21569177|
|    13030BA|21542906|
|    13024BA|21483391|
|    13022BA|21398939|
|    13027BA|21375556|
+-----------+--------+

按天分区耗时: 3.03 秒
按月分区耗时: 1.90 秒
按

In [0]:
%python
# 三.存储格式转换对比
#3.1 创建和写入ORC文件
orc_path = "/Volumes/data_catalog/retail_data/retail_volume/orc_data"

# 读取原始 CSV 数据
df_orders = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "UTF-8") \
    .csv("/Volumes/data_catalog/retail_data/retail_volume/orders_*.csv")

# 写入 ORC（覆盖）
df_orders.write.mode("overwrite").format("orc").save(orc_path)
print(f"ORC 数据已写入: {orc_path}")

# 创建一个临时视图用于查询
df_orders.createOrReplaceTempView("fact_order_orc_temp")
print("临时视图 fact_order_orc_temp 创建完成")

ORC 数据已写入: /Volumes/data_catalog/retail_data/retail_volume/orc_data
临时视图 fact_order_orc_temp 创建完成


In [0]:
%python
# 3.2 对比文件大小（Delta/Parquet vs ORC）
# Delta 表大小
delta_detail = spark.sql("DESCRIBE DETAIL fact_order_raw").collect()[0]
delta_size = delta_detail.sizeInBytes
delta_files = delta_detail.numFiles
print(f"Delta (Parquet) 表: 大小 = {delta_size / 1e6:.2f} MB, 文件数 = {delta_files}")

# ORC 文件大小
orc_files = dbutils.fs.ls(orc_path)
orc_total_size = sum(f.size for f in orc_files if not f.name.startswith('_'))
orc_file_count = len([f for f in orc_files if f.name.endswith('.orc')])
print(f"ORC 数据: 大小 = {orc_total_size / 1e6:.2f} MB, 文件数 = {orc_file_count}")
print(f"压缩比 (ORC/Parquet): {orc_total_size / delta_size:.2f}")

Delta (Parquet) 表: 大小 = 74.75 MB, 文件数 = 6
ORC 数据: 大小 = 72.00 MB, 文件数 = 9
压缩比 (ORC/Parquet): 0.96


In [0]:
%python
# 3.3 不同文件格式：查询效率对比（ORC vs delta）
# 预期结果：理论上ORC更快；但实际测试中，ORC比delta慢。我猜测可能和格式优化差异、谓词下推等因素相关。
import time
import statistics

def benchmark_query_delta(query_sql, runs=3):
    times = []
    for i in range(runs):
        start = time.time()
        spark.sql(query_sql).show()
        end = time.time()
        times.append(end - start)
        print(f"Delta 第{i+1}次: {end-start:.2f}秒")
    median = statistics.median(times)
    print(f"Delta 中位数耗时: {median:.2f}秒\n")
    return median

def benchmark_query_orc(query_sql, runs=3):
    times = []
    for i in range(runs):
        start = time.time()
        spark.sql(query_sql).show()
        end = time.time()
        times.append(end - start)
        print(f"ORC 第{i+1}次: {end-start:.2f}秒")
    median = statistics.median(times)
    print(f"ORC 中位数耗时: {median:.2f}秒\n")
    return median

# 查询语句
query_delta = """
    SELECT customer_id, SUM(amount) AS total
    FROM fact_order_raw
    GROUP BY customer_id
    ORDER BY total DESC
    LIMIT 10
"""

query_orc = """
    SELECT customer_id, SUM(amount) AS total
    FROM fact_order_orc_temp
    GROUP BY customer_id
    ORDER BY total DESC
    LIMIT 10
"""

print("=== 查询性能对比 (全表聚合) ===")
delta_time = benchmark_query_delta(query_delta, runs=3)
orc_time = benchmark_query_orc(query_orc, runs=3)

print(f"Delta (Parquet) 中位数: {delta_time:.2f}秒")
print(f"ORC 中位数: {orc_time:.2f}秒")
print(f"性能提升: {(1 - orc_time/delta_time)*100:.1f}%") #正数代表Delta快，负数代表ORC快。

=== 查询性能对比 (全表聚合) ===
+-----------+---------+
|customer_id|    total|
+-----------+---------+
|    13025BA|164090419|
|    13028BA|164017072|
|    13029BA|163768686|
|    13023BA|163506058|
|    13021BA|163448211|
|    13022BA|163336837|
|    13030BA|163304082|
|    13024BA|163282885|
|    13027BA|163258001|
|    13026BA|163230447|
+-----------+---------+

Delta 第1次: 0.91秒
+-----------+---------+
|customer_id|    total|
+-----------+---------+
|    13025BA|164090419|
|    13028BA|164017072|
|    13029BA|163768686|
|    13023BA|163506058|
|    13021BA|163448211|
|    13022BA|163336837|
|    13030BA|163304082|
|    13024BA|163282885|
|    13027BA|163258001|
|    13026BA|163230447|
+-----------+---------+

Delta 第2次: 0.88秒
+-----------+---------+
|customer_id|    total|
+-----------+---------+
|    13025BA|164090419|
|    13028BA|164017072|
|    13029BA|163768686|
|    13023BA|163506058|
|    13021BA|163448211|
|    13022BA|163336837|
|    13030BA|163304082|
|    13024BA|163282885|
|    1

In [0]:
%python
#四.数据倾斜与处理
# 4.1 检查数据倾斜程度
# 倾斜程度通过比较百分位数得出。例如，如果后99%客户的订单数远小于前1%的超级大客户，说明数据倾斜严重。
# 总数据：1000 万（行）、最大倾斜key数据=最大客户的订单数；
print("=== 订单量 Top 10 客户 ===")
spark.sql("""
    SELECT customer_id, COUNT(*) AS order_count
    FROM fact_order_raw
    GROUP BY customer_id
    ORDER BY order_count DESC
    LIMIT 10
""").show()

print("=== 订单量分布统计（百分位数）===")
spark.sql("""
    SELECT 
        percentile_approx(order_count, 0.5) AS median,
        percentile_approx(order_count, 0.9) AS p90,
        percentile_approx(order_count, 0.99) AS p99,
        max(order_count) AS max
    FROM (
        SELECT customer_id, COUNT(*) AS order_count
        FROM fact_order_raw
        GROUP BY customer_id
    )
""").show()

=== 订单量 Top 10 客户 ===
+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|    13025BA|     300660|
|    13022BA|     300229|
|    13028BA|     300191|
|    13029BA|     300039|
|    13023BA|     299652|
|    13021BA|     299650|
|    13030BA|     299622|
|    13026BA|     299608|
|    13024BA|     299431|
|    13027BA|     299235|
+-----------+-----------+

=== 订单量分布统计（百分位）===
+------+---+---+------+
|median|p90|p99|   max|
+------+---+---+------+
|   379|404|425|300660|
+------+---+---+------+



In [0]:
%python
# 4.2 测试倾斜情况下的查询效率
import time

print("=== 普通 GROUP BY 查询（按客户汇总金额）===")
sql_skew = """
    SELECT customer_id, SUM(amount) AS total_amount
    FROM fact_order_raw
    GROUP BY customer_id
    ORDER BY total_amount DESC
    LIMIT 10
"""

start = time.time()
spark.sql(sql_skew).show()
normal_time = time.time() - start
print(f"普通 GROUP BY 耗时: {normal_time:.2f} 秒")

=== 普通 GROUP BY 查询（按客户汇总金额）===
+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|    13025BA|   164090419|
|    13028BA|   164017072|
|    13029BA|   163768686|
|    13023BA|   163506058|
|    13021BA|   163448211|
|    13022BA|   163336837|
|    13030BA|   163304082|
|    13024BA|   163282885|
|    13027BA|   163258001|
|    13026BA|   163230447|
+-----------+------------+

普通 GROUP BY 耗时: 0.79 秒


In [0]:
%python
# 4.3：两阶段聚合优化
# 步骤：
# 1. 添加随机前缀（0~N-1），打散倾斜 key
# 2. 先按 (random_prefix, customer_id) 局部聚合
# 3. 去掉前缀，再按 customer_id 全局聚合
# 4. 对比效率

# 设置打散因子（可根据倾斜程度调整，此处设为 10）
shuffle_factor = 10

sql_optimized = f"""
    WITH tmp AS (
        SELECT 
            CONCAT(CAST(FLOOR(RAND() * {shuffle_factor}) AS STRING), '_', customer_id) AS salted_key,
            customer_id,
            amount
        FROM fact_order_raw
    ),
    partial_agg AS (
        SELECT 
            salted_key,
            customer_id,
            SUM(amount) AS partial_sum
        FROM tmp
        GROUP BY salted_key, customer_id
    )
    SELECT 
        customer_id,
        SUM(partial_sum) AS total_amount
    FROM partial_agg
    GROUP BY customer_id
    ORDER BY total_amount DESC
    LIMIT 10
"""

print(f"=== 两阶段聚合优化（打散因子={shuffle_factor}）===")
start = time.time()
spark.sql(sql_optimized).show()
optimized_time = time.time() - start
print(f"优化后耗时: {optimized_time:.2f} 秒")
print(f"性能提升: {(1 - optimized_time/normal_time)*100:.1f}%")

=== 两阶段聚合优化（打散因子=10）===
+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|    13025BA|   164090419|
|    13028BA|   164017072|
|    13029BA|   163768686|
|    13023BA|   163506058|
|    13021BA|   163448211|
|    13022BA|   163336837|
|    13030BA|   163304082|
|    13024BA|   163282885|
|    13027BA|   163258001|
|    13026BA|   163230447|
+-----------+------------+

优化后耗时: 0.83 秒
性能提升: -6.0%


In [0]:
%python
# 4.4：对比优化前后执行计划（如果出现无效优化/负优化，则需要分析计划，人工查看问题出在哪）
print("=== 普通 GROUP BY 执行计划（部分）===")
spark.sql(sql_skew).explain(False)

print("\n=== 两阶段聚合执行计划（部分）===")
spark.sql(sql_optimized).explain(False)

=== 普通 GROUP BY 执行计划（部分）===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonTopK(sortOrder=[total_amount#21014L DESC NULLS LAST], partitionOrderCount=0)
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#22953]
               +- PhotonShuffleExchangeSink SinglePartition
                  +- PhotonTopK(sortOrder=[total_amount#21014L DESC NULLS LAST], partitionOrderCount=0)
                     +- PhotonGroupingAgg(keys=[customer_id#21021], functions=[finalmerge_sum(merge sum#21029L) AS sum(amount)#21025L])
                        +- PhotonShuffleExchangeSource
                           +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#22945]
                              +- PhotonShuffleExchangeSink hashpartitioning(customer_id#21021, 114)
                                 +- PhotonGroupingAgg(keys=[customer_id#21021], functions=[partial_sum(am

In [0]:
%python
# 4.5 MapJoin 优化示例
# MapJoin适用于事实表（如 order）与维度表（如 product、customer）的场景
# 本项目主要是单表聚合，因此此处仅供演示。该查询运行不一定更快。
print("=== MapJoin 示例：订单表 join 产品表（小表广播）===")
sql_mapjoin = """
    SELECT /*+ MAPJOIN(dp) */ 
        fo.customer_id,
        dp.product_category,
        SUM(fo.amount) AS total_amount
    FROM fact_order_raw fo
    JOIN dim_product dp ON fo.product_id = dp.product_id
    GROUP BY fo.customer_id, dp.product_category
    ORDER BY total_amount DESC
    LIMIT 10
"""
start = time.time()
spark.sql(sql_mapjoin).show()
mapjoin_time = time.time() - start
print(f"MapJoin 耗时: {mapjoin_time:.2f} 秒")

=== MapJoin 示例：订单表 join 产品表（小表广播）===
+-----------+----------------+------------+
|customer_id|product_category|total_amount|
+-----------+----------------+------------+
|    13025BA|              球|   120781520|
|    13028BA|              球|   120720611|
|    13029BA|              球|   120515340|
|    13023BA|              球|   120364685|
|    13021BA|              球|   120295007|
|    13022BA|              球|   120177010|
|    13030BA|              球|   120169354|
|    13026BA|              球|   120164043|
|    13027BA|              球|   120007794|
|    13024BA|              球|   119999343|
+-----------+----------------+------------+

MapJoin 耗时: 2.15 秒
